# `StructurePreprocessor.encode_dssp()`

`encode_dssp` runs DSSP on each entry's structure file and encodes the per-residue secondary structure, solvent accessibility, backbone dihedrals and hydrogen bonds into a `[0, 1]`-normalized `dict_num` (`{entry: (L, D)}`) for `CPP.run_num`. `features` is any subset of `{ss3, ss8, rasa, phi_psi_sincos, hbond_donor, hbond_acceptor}`; the channels are concatenated along `D` in the order given.

Requires `aaanalysis[pro]` plus a `mkdssp` binary on `PATH`. The structures are AlphaFold models fetched with `fetch_alphafold`.

In [1]:
import warnings
import tempfile
from pathlib import Path
import aaanalysis as aa
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

# Three short human proteins from the bundled gamma-secretase set; their ``entry``
# values are UniProt accessions, which is what AlphaFold DB is keyed by.
df_seq = aa.load_dataset(name='DOM_GSEC', n=10)
df_seq = df_seq[df_seq['entry'].isin(['Q14802', 'O43914', 'P01135'])].reset_index(drop=True)

strp = aa.StructurePreprocessor(verbose=False)
af_dir = Path(tempfile.mkdtemp()) / 'alphafold'   # use a persistent project folder in real work

strp.fetch_alphafold(df_seq=df_seq, out_folder=af_dir)

features = ['ss3', 'rasa', 'phi_psi_sincos']       # 3 + 1 + 4 = 8 channels
dict_dssp = strp.encode_dssp(df_seq=df_seq, pdb_folder=af_dir, features=features)
print({entry: arr.shape for entry, arr in dict_dssp.items()})

{'Q14802': (87, 8), 'P01135': (160, 8), 'O43914': (113, 8)}


The first three channels are the one-hot three-state secondary structure, so their column means are the helix / strand / coil fractions of each protein; `build_cat` names all eight channels for `CPP`:

In [2]:
import numpy as np
import pandas as pd

df_cat = strp.build_cat(features=features)
df_ss = pd.DataFrame({entry: np.nanmean(arr[:, :3], axis=0) for entry, arr in dict_dssp.items()},
                     index=df_cat['scale_id'][:3]).T.round(2)
aa.display_df(df_ss, n_rows=10, show_shape=True)

DataFrame shape: (3, 3)


scale_id,ss_helix,ss_strand,ss_coil
Q14802,0.790000,0.000000,0.210000
P01135,0.570000,0.140000,0.290000
O43914,0.710000,0.000000,0.290000


## Further parameters

`ss_mode` and `gap_handling` are forwarded to `get_dssp` when DSSP runs inline (`'ss8'` for the eight-state alphabet, `'omit'` to drop residues DSSP skips); `on_failure` decides what happens to entries whose DSSP run fails (`'nan'` fills a NaN tensor, `'drop'` removes them, `'raise'` raises); `return_df=True` also returns the per-row status frame with an `encode_dssp_ok` column.

In [3]:
dict_dssp8, df_status = strp.encode_dssp(df_seq=df_seq, pdb_folder=af_dir,
                                         features=['ss8', 'hbond_donor', 'hbond_acceptor'],
                                         ss_mode='ss8', gap_handling='omit', on_failure='nan', return_df=True)
print({entry: arr.shape for entry, arr in dict_dssp8.items()})
aa.display_df(df_status[['entry', 'gene', 'dssp_ok', 'encode_dssp_ok']], n_rows=10, show_shape=True)

{'Q14802': (57, 12), 'P01135': (129, 12), 'O43914': (80, 12)}
DataFrame shape: (3, 4)


,entry,gene,dssp_ok,encode_dssp_ok
1,Q14802,FXYD3,True,True
2,P01135,TGFA,True,True
3,O43914,TYROBP,True,True
